# ARCHIVED — Mistral Foundry early pipeline

**Do not use for day-to-day extraction.** Kept only as history.

Current notebooks:
- `notebooks/01-mistral-ocr-and-fields.ipynb`
- `notebooks/02-content-understanding-tables.ipynb`

See `README.md`.


# Mistral Document AI — Azure Foundry
EN 10204 material certificate extractor.

**Pipeline per document:**
1. OCR page 0 only → detect cert type (2.1 / 2.2 / 3.1 / 3.2) and primary cert page count
2. Gate on cert type — 2.1/2.2 carry no test data, skip structured extraction
3. For 3.1/3.2: OCR remaining cert pages (up to detected page count, max 3)
4. Validate expected fields are present; flag anything missing
5. Save markdown + JSON to `extraction output/`

## 0. Setup

In [ ]:
import base64
import json
import os
import re
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

_raw = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT", "").rstrip("/")
_parsed = urlparse(_raw)
OCR_ENDPOINT = f"{_parsed.scheme}://{_parsed.netloc}/providers/mistral/azure/ocr"
KEY          = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_KEY")
DEPLOYMENT   = os.getenv("AZURE_AI_DEPLOYMENT_NAME")
HEADERS      = {"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"}

INPUT_DIR  = Path("raw docs")
OUTPUT_DIR = Path("extraction output")

# Subfolders are named by model/version so runs can be compared side-by-side later.
# Add a new subfolder (e.g. "ocr-mistral-ocr-2505") when testing a different OCR model.
OCR_DIR        = OUTPUT_DIR / f"ocr-{DEPLOYMENT}"   # raw OCR markdown + Mistral JSON
STRUCTURED_DIR = OUTPUT_DIR / "structured-v1"        # field extraction output; bump to v2 when prompt changes

for d in [OUTPUT_DIR, OCR_DIR, STRUCTURED_DIR]:
    d.mkdir(exist_ok=True)

print("OCR endpoint   :", OCR_ENDPOINT)
print("Deployment     :", DEPLOYMENT)
print("OCR output     :", OCR_DIR.resolve())
print("Structured out :", STRUCTURED_DIR.resolve())

## 1. OCR helpers

In [ ]:
def encode_file(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode()


def ocr_pages(path: Path, pages: list[int], include_images: bool = False) -> dict:
    """
    OCR specific pages of a PDF via Mistral Document AI on Azure Foundry.
    Passing a pages list avoids billing for enclosure pages (UT reports,
    hardness worksheets, NDE rosters) that contain no EN 10204 cert data.
    """
    payload = {
        "model": DEPLOYMENT,
        "document": {
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{encode_file(path)}",
        },
        "include_image_base64": include_images,
        "pages": pages,
    }
    resp = requests.post(url=OCR_ENDPOINT, json=payload, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def merge_ocr_results(*results: dict) -> dict:
    """Combine pages from multiple OCR calls into a single result dict."""
    merged_pages = [page for r in results for page in r["pages"]]
    return {**results[0], "pages": merged_pages}


def pages_to_markdown(result: dict) -> str:
    return "\n\n---\n\n".join(p["markdown"] for p in result["pages"])


def save_ocr(pdf_path: Path, result: dict) -> tuple[Path, Path]:
    """
    Saves raw OCR output to OCR_DIR (named by deployment).
    Keeping OCR output separate from structured extraction lets you re-run
    the extraction step with a different prompt without re-calling the OCR API.
    """
    stem      = pdf_path.stem
    md_path   = OCR_DIR / f"{stem}.md"
    json_path = OCR_DIR / f"{stem}.json"
    md_path.write_text(pages_to_markdown(result), encoding="utf-8")
    json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return md_path, json_path


def save_structured(pdf_path: Path, data: dict) -> Path:
    """Saves structured field extraction output to STRUCTURED_DIR (versioned folder)."""
    json_path = STRUCTURED_DIR / f"{pdf_path.stem}.json"
    json_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    return json_path

## 2. Cert type detection

**Why detect cert type before parsing?**

EN 10204 defines 4 certificate types with fundamentally different content:

| Type | Contains test data? | Impact on parsing |
|------|--------------------|-----------------|
| 2.1 | No — declaration only | Skip extraction entirely. No chemistry, no mechanical properties to find. |
| 2.2 | Partial — non-specific (batch/typical values, not heat-specific) | Extract with low confidence. Values may not match the physical material supplied. |
| 3.1 | Yes — actual test results for the specific heat supplied | Full extraction. Validate that chemistry + mechanical + heat number are all present. |
| 3.2 | Yes — same as 3.1, also witnessed by independent 3rd party | Full extraction + flag 3rd party signatory. |

Knowing the type before parsing means:
- **No wasted extraction** on 2.1/2.2 docs
- **Field validation** — a 3.1 cert missing chemistry is a red flag (incomplete or potentially fraudulent)
- **Correct page count** — 3.1/3.2 always have at least 1 primary cert page; detection drives how many pages to OCR
- **Confidence scoring** — downstream consumers know whether values are heat-specific or just typical

In [ ]:
# Matches all real-world EN 10204 type declarations seen in the wild:
#   "EN 10204 3.1", "BSEN 10204:2004 3.1", "DIN EN 10204/01.05 3.1"
#   "INSPECTION CERTIFICATE TYPE: EN 10204 3.1"
_CERT_TYPE_RE = re.compile(
    r'(?:BS\s*)?(?:DIN\s+)?EN\s*10204[^0-9]*([23]\.[12])',
    re.IGNORECASE
)

# Matches page count Y from markers like:
#   "Page: 1/2", "1/1", "Pag 1 of 2", "Sheet 1/1", "page 1 of 2", "1 de 2"
_PAGE_COUNT_RE = re.compile(
    r'(?:page|pag|sheet|foglio|p\.)?[\s:]*1\s*(?:/|of|de)\s*(\d+)',
    re.IGNORECASE
)

# EN 10204 primary certs are never more than 3 pages.
# Guards against malformed page markers pulling in enclosure pages.
_MAX_CERT_PAGES = 3


def detect_cert_type(text: str) -> str | None:
    """
    Returns the EN 10204 type string (e.g. '3.1') or None if not found.
    None means either the doc is not an EN 10204 cert or page 0 OCR failed.
    """
    match = _CERT_TYPE_RE.search(text)
    return match.group(1) if match else None


def detect_cert_page_count(text: str) -> int:
    """
    Returns how many pages belong to the primary cert document.

    Falls back to 2 when no marker is found — this deliberately covers the
    distributor-wrap pattern (e.g. Howco cover page + VDM mill cert) where
    the cover page carries no page numbering of its own but the mill cert
    starts on the next page.
    """
    match = _PAGE_COUNT_RE.search(text)
    if match:
        return min(int(match.group(1)), _MAX_CERT_PAGES)
    return 2  # safe default for wrapped/multi-party certs


# Fields that MUST be present in a valid 3.1/3.2 cert.
# Absence of any of these is a non-conformance — flag, don't silently skip.
_REQUIRED_31_PATTERNS = {
    "heat_number":    re.compile(r'(?:heat|cast|coulée|colata)\s*(?:no\.?|nr\.?|number)?\s*[:\-]?\s*([A-Z0-9]{4,})', re.IGNORECASE),
    "chemical_analysis": re.compile(r'(?:chemical\s*anal|analisi\s*chim|composition\s*chim|cast\s*anal)', re.IGNORECASE),
    "mechanical_props":  re.compile(r'(?:tensile|mechanical\s*prop|prove\s*meccan|propiedades\s*mec)', re.IGNORECASE),
    "heat_treatment":    re.compile(r'(?:heat\s*treat|trattament|tratamiento\s*t)', re.IGNORECASE),
}


def validate_31_fields(full_text: str) -> dict[str, bool]:
    """
    Checks whether each mandatory EN 10204 3.1 field is detectable in the
    extracted text. Returns a presence map; False entries should be flagged
    as non-conformances, not silently ignored.
    """
    return {
        field: bool(pattern.search(full_text))
        for field, pattern in _REQUIRED_31_PATTERNS.items()
    }

## 3. Extraction pipeline

In [ ]:
def process_cert(pdf_path: Path) -> dict:
    """
    Two-step pipeline that minimises API cost by using cert type and page
    count — both detected from page 0 — to fetch only the pages that
    contain EN 10204 data.

    Step 1  OCR page 0 only.
            Detect cert type. For 2.1/2.2 this is the only API call made.
            Detect primary cert page count to scope step 2.

    Step 2  OCR remaining cert pages (only for 3.1/3.2).
            Pages are bounded by the detected count, capped at 3.
            Distributor covers with no page marker default to 2 pages
            so the embedded mill cert (always on page 1) is captured.

    Returns a result dict containing:
        cert_type     detected EN 10204 type string or None
        page_count    number of cert pages OCR'd
        skip_reason   set when 2.1/2.2 or unknown type — no extraction done
        validation    field presence map for 3.1/3.2 certs
        missing       list of mandatory fields not found (non-conformances)
        ocr           raw Mistral response (merged pages)
    """
    print(f"  [1/2] OCR page 0 — {pdf_path.name}")
    page0_result = ocr_pages(pdf_path, pages=[0])
    page0_text   = page0_result["pages"][0]["markdown"]

    cert_type  = detect_cert_type(page0_text)
    page_count = detect_cert_page_count(page0_text)

    print(f"      cert type : {cert_type or 'not detected'}")
    print(f"      cert pages: {page_count}")

    # 2.1 / 2.2 — declaration certs carry no heat-specific test data.
    # Extracting fields would yield nothing meaningful; flag and stop.
    if cert_type in ("2.1", "2.2"):
        return {
            "cert_type":   cert_type,
            "page_count":  1,
            "skip_reason": f"EN 10204 {cert_type} — declaration cert, no test results present",
            "ocr":         page0_result,
        }

    # Unknown type — could be a non-cert document or OCR failure on page 0.
    if cert_type is None:
        return {
            "cert_type":   None,
            "page_count":  1,
            "skip_reason": "EN 10204 type not detected on page 0 — verify document",
            "ocr":         page0_result,
        }

    # 3.1 / 3.2 — fetch remaining cert pages if the cert spans more than page 0.
    if page_count > 1:
        remaining_pages = list(range(1, page_count))
        print(f"  [2/2] OCR pages {remaining_pages}")
        rest_result = ocr_pages(pdf_path, pages=remaining_pages)
        final_ocr   = merge_ocr_results(page0_result, rest_result)
    else:
        print(f"  [2/2] Single-page cert — no additional OCR needed")
        final_ocr = page0_result

    full_text  = pages_to_markdown(final_ocr)
    validation = validate_31_fields(full_text)
    missing    = [f for f, present in validation.items() if not present]

    if missing:
        print(f"  ⚠  Missing mandatory fields: {missing}")
    else:
        print(f"  ✓  All mandatory EN 10204 3.1 fields detected")

    return {
        "cert_type":  cert_type,
        "page_count": page_count,
        "validation": validation,
        "missing":    missing,
        "ocr":        final_ocr,
    }

## 4. List available PDFs

In [ ]:
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
for i, p in enumerate(pdf_files):
    print(f"[{i}] {p.name}  ({p.stat().st_size // 1024} KB)")

## 5. Process a single file
Change `FILE_INDEX` to pick a different PDF.

In [ ]:
FILE_INDEX = 0  # ← change to 1 or 2 for other files

target = pdf_files[FILE_INDEX]
print(f"\nFile: {target.name}  ({target.stat().st_size // 1024} KB)\n")

cert = process_cert(target)

if "skip_reason" in cert:
    print(f"\nSkipped: {cert['skip_reason']}")
else:
    md_path, json_path = save_ocr(target, cert["ocr"])
    print(f"\nSaved OCR → {md_path.relative_to(OUTPUT_DIR)}")
    print(f"           {json_path.relative_to(OUTPUT_DIR)}")

print("\nSummary:", json.dumps({k: v for k, v in cert.items() if k != "ocr"}, indent=2))

## 6. Preview extracted content

In [ ]:
if "skip_reason" not in cert:
    for i, page in enumerate(cert["ocr"]["pages"]):
        display(Markdown(f"### Page {i+1}"))
        display(Markdown(page["markdown"]))
else:
    print(cert["skip_reason"])

## 7. Batch — process all PDFs
**Only run this cell when you want to process all files. Each file makes 1–2 API calls.**

In [ ]:
for pdf in pdf_files:
    print(f"\n{'─'*60}")
    print(f"FILE: {pdf.name}")
    print('─'*60)
    try:
        cert = process_cert(pdf)
        if "skip_reason" in cert:
            print(f"  Skipped: {cert['skip_reason']}")
        else:
            md_p, json_p = save_ocr(pdf, cert["ocr"])
            print(f"  OCR saved → {md_p.relative_to(OUTPUT_DIR)}, {json_p.relative_to(OUTPUT_DIR)}")
    except requests.HTTPError as e:
        print(f"  API ERROR {e.response.status_code}: {e.response.text[:200]}")

## 8. Structured field extraction

Converts OCR markdown into a typed JSON record using a chat model.

**Separation of concerns:**
- OCR (Mistral Document AI) → converts image/PDF to text. Expensive per page.
- Extraction (chat model) → converts text to structured fields. Cheap per token.

Keeping them separate means you can re-run extraction with an improved prompt
without re-calling the OCR API. The OCR output saved in `ocr-<model>/` is the
stable input; `structured-v1/` is the versioned output. Bump to `structured-v2`
when the prompt or schema changes.

**Required `.env` additions:**
```
AZURE_CHAT_ENDPOINT=https://<resource>.services.ai.azure.com/providers/mistral/azure/v1/chat/completions
AZURE_CHAT_KEY=<key>
AZURE_CHAT_DEPLOYMENT=mistral-small-2503
```
Copy endpoint and key from the deployment page in Azure Foundry, same as you did for the OCR model.
Keeping these explicit means you can point at a different resource, model provider, or even OpenAI directly without touching the code.

In [ ]:
CHAT_ENDPOINT   = os.getenv("AZURE_CHAT_ENDPOINT", "")
CHAT_KEY        = os.getenv("AZURE_CHAT_KEY", "")
CHAT_DEPLOYMENT = os.getenv("AZURE_CHAT_DEPLOYMENT", "")
CHAT_HEADERS    = {"Content-Type": "application/json", "Authorization": f"Bearer {CHAT_KEY}"}

print("Chat endpoint :", CHAT_ENDPOINT  or "⚠  not set — add AZURE_CHAT_ENDPOINT to .env")
print("Chat model    :", CHAT_DEPLOYMENT or "⚠  not set — add AZURE_CHAT_DEPLOYMENT to .env")
print("Chat key set  :", bool(CHAT_KEY))

In [ ]:
# Prompt is parameterised by cert_type so the model knows exactly which fields
# to expect and can flag absence as missing rather than hallucinating a value.
_SYSTEM_PROMPT = """\
You are a material certificate data extractor specialising in EN 10204 inspection certificates.
Extract ONLY values explicitly present in the provided text. Do NOT infer, calculate, or hallucinate.
If a field is not present, use null. Return valid JSON only — no explanation, no markdown fences.\
"""

_EXTRACTION_SCHEMA = """\
{
  "cert_number": "",
  "cert_type": "",
  "issue_date": "",
  "issuer": "",
  "customer": "",
  "purchase_order": "",
  "product": {
    "form": "",
    "grade": "",
    "uns": "",
    "specification": [],
    "dimensions": "",
    "quantity": null,
    "heat_numbers": []
  },
  "melt_practice": "",
  "chemical_analysis": [
    {"heat": "", "C": null, "Si": null, "Mn": null, "P": null, "S": null,
     "Cr": null, "Ni": null, "Mo": null, "N": null, "Nb_Ta": null, "Co": null, "Fe": null}
  ],
  "mechanical_properties": [
    {"heat": "", "test_standard": "", "temp_c": null,
     "yield_mpa": null, "uts_mpa": null, "elongation_pct": null, "reduction_area_pct": null}
  ],
  "impact_test": [
    {"heat": "", "standard": "", "temp_c": null, "energy_j": []}
  ],
  "hardness": [
    {"heat": "", "method": "", "values": []}
  ],
  "heat_treatment": {
    "type": "", "temp_c": null, "soak": "", "cooling": ""
  },
  "grain_size": null,
  "conforming": null,
  "signatory": "",
  "third_party_inspector": ""
}\
"""


def build_extraction_prompt(cert_markdown: str, cert_type: str) -> str:
    # cert_type drives field expectations: 3.2 requires third_party_inspector,
    # 2.2 values are non-specific so the model is warned not to over-trust them.
    type_note = {
        "3.1": "This is a 3.1 cert — all values are heat-specific and traceable.",
        "3.2": "This is a 3.2 cert — heat-specific values, also extract third_party_inspector.",
        "2.2": "This is a 2.2 cert — values may be batch-typical, NOT heat-specific. Set cert_type to '2.2'.",
        "2.1": "This is a 2.1 cert — no test data present. Return the schema with all test fields as null.",
    }.get(cert_type, "Cert type unknown — extract whatever is present.")

    return (
        f"{type_note}\n\n"
        f"Return a JSON object matching this schema exactly:\n{_EXTRACTION_SCHEMA}\n\n"
        f"Certificate text:\n\n{cert_markdown}"
    )


def extract_structured(cert_markdown: str, cert_type: str) -> dict:
    """
    Sends OCR markdown to a chat model for structured field extraction.
    Using a separate cheap chat model (not Document AI) keeps this step
    inexpensive — the OCR cost is already paid.
    """
    if not CHAT_ENDPOINT or not CHAT_KEY:
        raise RuntimeError("AZURE_CHAT_ENDPOINT and AZURE_CHAT_KEY must be set in .env")

    payload = {
        "model": CHAT_DEPLOYMENT,
        "messages": [
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user",   "content": build_extraction_prompt(cert_markdown, cert_type)},
        ],
        "temperature": 0,     # deterministic — extraction, not generation
        "response_format": {"type": "json_object"},
    }
    resp = requests.post(url=CHAT_ENDPOINT, json=payload, headers=CHAT_HEADERS)
    resp.raise_for_status()
    raw = resp.json()["choices"][0]["message"]["content"]
    return json.loads(raw)

### 8a. Extract structured fields — single file
Reads from the OCR markdown already saved in `ocr-<model>/`. No OCR API call made here.

In [ ]:
FILE_INDEX = 0  # ← must match the file processed in Section 5

target      = pdf_files[FILE_INDEX]
ocr_md_path = OCR_DIR / f"{target.stem}.md"

if not ocr_md_path.exists():
    print(f"⚠  OCR output not found: {ocr_md_path}")
    print("   Run Section 5 first to generate the OCR markdown.")
else:
    cert_markdown = ocr_md_path.read_text(encoding="utf-8")

    # cert_type comes from process_cert() run in Section 5.
    # If re-running standalone, re-detect from the saved markdown.
    cert_type = detect_cert_type(cert_markdown) or "3.1"

    print(f"Extracting from : {ocr_md_path.relative_to(OUTPUT_DIR)}")
    print(f"Cert type       : {cert_type}")
    print(f"Chat model      : {CHAT_DEPLOYMENT}\n")

    structured = extract_structured(cert_markdown, cert_type)
    out_path   = save_structured(target, structured)

    print(f"Saved → {out_path.relative_to(OUTPUT_DIR)}")
    display(Markdown("```json\n" + json.dumps(structured, indent=2, ensure_ascii=False) + "\n```"))

### 8b. Batch structured extraction — all files with saved OCR output
Iterates `ocr-<model>/` folder. Only processes files that have OCR output saved.

In [ ]:
for md_file in sorted(OCR_DIR.glob("*.md")):
    print(f"\n{'─'*60}")
    print(f"FILE: {md_file.name}")
    print('─'*60)
    try:
        cert_markdown = md_file.read_text(encoding="utf-8")
        cert_type     = detect_cert_type(cert_markdown) or "3.1"
        structured    = extract_structured(cert_markdown, cert_type)
        # Reconstruct pdf_path just for naming — file doesn't need to exist
        out_path      = save_structured(INPUT_DIR / f"{md_file.stem}.pdf", structured)
        print(f"  {cert_type} → {out_path.relative_to(OUTPUT_DIR)}")
    except Exception as e:
        print(f"  ERROR: {e}")